# 📊 진동 데이터 분석

> **학습 목표**: KAMP 진동 센서 데이터를 분석하여 설비 상태를 파악합니다.

---

## 📋 학습 내용

1. ✅ 시계열 진동 데이터 로드 및 시각화
2. ✅ 통계적 특징 추출 (RMS, Peak, Kurtosis)
3. ✅ 축별 진동 비교
4. ✅ 온도-진동 상관관계
5. ✅ 이상 징후 탐지

**소요 시간**: 약 45분  
**난이도**: ⭐⭐ (중급)

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 데이터 분석
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ 라이브러리 로드 완료!")

## 📂 Step 2: 진동 데이터 로드

In [ ]:
# 데이터 파일 경로
data_path = Path('../data/sample_kamp_vibration.csv')

# 데이터 로드
def load_data(file_path):
    encodings = ['utf-8-sig', 'cp949', 'utf-8']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ 데이터 로드 성공! (인코딩: {encoding})")
            return df
        except:
            continue
    raise ValueError("데이터 로드 실패")

df = load_data(data_path)
print(f"\n📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"💾 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 데이터 미리보기
df.head()

In [ ]:
# 데이터 구조 파악
print("📋 컬럼 정보:")
df.info()

print("\n📊 기초 통계:")
df.describe()

## 📈 Step 3: 시계열 진동 데이터 시각화

In [ ]:
# 진동 관련 컬럼 찾기
vibration_cols = [col for col in df.columns if any(keyword in col.lower() 
                  for keyword in ['vib', 'x', 'y', 'z', '진동'])]

print(f"📊 진동 관련 컬럼: {len(vibration_cols)}개")
print(vibration_cols)

# 시간 컬럼 찾기
time_cols = [col for col in df.columns if any(keyword in col.lower() 
             for keyword in ['time', 'timestamp', 'date', '시간'])]
print(f"\n⏰ 시간 관련 컬럼: {time_cols}")

In [ ]:
# 시계열 플롯 (처음 1000개 샘플)
if len(vibration_cols) >= 3:
    sample_size = min(1000, len(df))
    df_sample = df.iloc[:sample_size]

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))

    for idx, col in enumerate(vibration_cols[:3]):
        axes[idx].plot(df_sample[col], linewidth=0.8)
        axes[idx].set_title(f'{col} 진동', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('진폭')
        axes[idx].grid(alpha=0.3)

    axes[2].set_xlabel('샘플')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 진동 데이터 컬럼을 찾을 수 없습니다.")

## 🔢 Step 4: 통계적 특징 추출

### 진동 분석 주요 지표:
- **RMS (Root Mean Square)**: 전체 진동 에너지
- **Peak-to-Peak**: 최대 진폭
- **Kurtosis (첨도)**: 충격성 진동 감지
- **Crest Factor**: RMS 대비 Peak 비율

In [ ]:
# 특징 추출 함수
def calculate_rms(signal):
    """RMS 계산"""
    return np.sqrt(np.mean(signal**2))

def calculate_peak_to_peak(signal):
    """Peak-to-Peak 계산"""
    return np.max(signal) - np.min(signal)

def calculate_kurtosis(signal):
    """Kurtosis (첨도) 계산"""
    mean = np.mean(signal)
    std = np.std(signal)
    return np.mean(((signal - mean) / std) ** 4)

def calculate_crest_factor(signal):
    """Crest Factor 계산"""
    return np.max(np.abs(signal)) / calculate_rms(signal)

# 모든 진동 컬럼에 대해 특징 추출
features = {}

for col in vibration_cols[:3]:  # 처음 3개만
    signal = df[col].dropna()
    features[col] = {
        'RMS': calculate_rms(signal),
        'Peak-to-Peak': calculate_peak_to_peak(signal),
        'Kurtosis': calculate_kurtosis(signal),
        'Crest Factor': calculate_crest_factor(signal),
        'Mean': signal.mean(),
        'Std': signal.std()
    }

# 결과 DataFrame
features_df = pd.DataFrame(features).T
print("📊 진동 특징 추출 결과:")
features_df

In [ ]:
# 특징 비교 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['RMS', 'Peak-to-Peak', 'Kurtosis', 'Crest Factor']
for idx, metric in enumerate(metrics):
    ax = axes[idx//2, idx%2]
    features_df[metric].plot(kind='bar', ax=ax, color='skyblue')
    ax.set_title(f'{metric} 비교', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n💡 해석 가이드:")
print("   - RMS 높음: 전체 진동 에너지 증가")
print("   - Kurtosis > 3: 충격성 진동 존재")
print("   - Crest Factor > 5: 불규칙한 충격")

## 📊 Step 5: 진동 분포 분석

In [ ]:
# 히스토그램 및 박스플롯
if len(vibration_cols) >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 히스토그램
    for col in vibration_cols[:3]:
        axes[0].hist(df[col].dropna(), bins=50, alpha=0.5, label=col)
    axes[0].set_title('진동 분포 (히스토그램)', fontweight='bold')
    axes[0].set_xlabel('진폭')
    axes[0].set_ylabel('빈도')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # 박스플롯
    df[vibration_cols[:3]].plot(kind='box', ax=axes[1])
    axes[1].set_title('진동 분포 (박스플롯)', fontweight='bold')
    axes[1].set_ylabel('진폭')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## 🌡️ Step 6: 온도-진동 상관관계

In [ ]:
# 온도 컬럼 찾기
temp_cols = [col for col in df.columns if any(keyword in col.lower() 
             for keyword in ['temp', '온도', 'temperature'])]

if temp_cols and vibration_cols:
    print(f"🌡️ 온도 컬럼: {temp_cols}")

    # 상관관계 분석
    corr_cols = vibration_cols[:3] + temp_cols[:1]
    corr_matrix = df[corr_cols].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title('온도-진동 상관관계', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\n📊 상관계수 해석:")
    print("   |r| > 0.7: 강한 상관관계")
    print("   |r| > 0.4: 중간 상관관계")
    print("   |r| < 0.3: 약한 상관관계")
else:
    print("⚠️ 온도 또는 진동 데이터를 찾을 수 없습니다.")

## 🚨 Step 7: 이상 징후 탐지

In [ ]:
# 통계적 이상 탐지 (3-sigma 방법)
def detect_anomalies(signal, threshold=3):
    """3-sigma 방법으로 이상치 탐지"""
    mean = np.mean(signal)
    std = np.std(signal)
    upper_bound = mean + threshold * std
    lower_bound = mean - threshold * std

    anomalies = (signal > upper_bound) | (signal < lower_bound)
    return anomalies, upper_bound, lower_bound

if vibration_cols:
    # 첫 번째 진동 컬럼 분석
    col = vibration_cols[0]
    signal = df[col].dropna()

    anomalies, upper, lower = detect_anomalies(signal)
    anomaly_count = anomalies.sum()
    anomaly_pct = (anomaly_count / len(signal)) * 100

    print(f"🔍 이상 징후 탐지 결과 ({col}):")
    print(f"   총 샘플: {len(signal):,}개")
    print(f"   이상치: {anomaly_count:,}개 ({anomaly_pct:.2f}%)")
    print(f"   정상 범위: [{lower:.3f}, {upper:.3f}]")

    # 시각화
    sample_size = min(1000, len(signal))
    plt.figure(figsize=(14, 6))
    plt.plot(signal.iloc[:sample_size].values, label='진동 신호', linewidth=0.8)
    plt.axhline(y=upper, color='r', linestyle='--', label='상한 (3σ)')
    plt.axhline(y=lower, color='r', linestyle='--', label='하한 (3σ)')
    plt.fill_between(range(sample_size), lower, upper, alpha=0.2, color='green')

    # 이상치 표시
    anomaly_indices = anomalies.iloc[:sample_size][anomalies.iloc[:sample_size]].index
    if len(anomaly_indices) > 0:
        plt.scatter(anomaly_indices, signal.loc[anomaly_indices], 
                   color='red', s=50, label='이상치', zorder=5)

    plt.title(f'{col} 이상 징후 탐지', fontsize=14, fontweight='bold')
    plt.xlabel('샘플')
    plt.ylabel('진폭')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 💾 Step 8: 분석 결과 저장

In [ ]:
# 출력 폴더 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 특징 저장
features_file = output_dir / '01_vibration_features.csv'
features_df.to_csv(features_file, encoding='utf-8-sig')
print(f"✅ 특징 저장: {features_file}")

# 이상치 로그 저장
if 'anomalies' in locals():
    anomaly_log = pd.DataFrame({
        'index': anomalies[anomalies].index,
        'value': signal[anomalies]
    })
    anomaly_file = output_dir / '01_anomalies.csv'
    anomaly_log.to_csv(anomaly_file, index=False, encoding='utf-8-sig')
    print(f"✅ 이상치 로그 저장: {anomaly_file}")

print("\n🎉 진동 분석 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. KAMP 진동 데이터 로드 및 시각화
2. RMS, Peak, Kurtosis 등 특징 추출
3. 축별 진동 비교 분석
4. 온도-진동 상관관계 분석
5. 3-sigma 방법으로 이상 징후 탐지

### 💡 핵심 인사이트
- RMS와 Kurtosis는 설비 상태의 주요 지표
- 온도와 진동의 상관관계 파악 중요
- 통계적 방법으로 이상 징후 조기 발견

### 📚 다음 단계
- **02_fft_frequency_analysis.ipynb**: FFT를 통한 주파수 분석
- **03_anomaly_detection.ipynb**: 머신러닝 기반 이상 탐지

---

*제조AI 교육 v12 Enhanced | Part 2-1 | 2025.02*